# 04 · Evaluación e interpretación de los clústeres

**Entradas:** `datos/procesados/resenas_limpias.csv` + modelos de `modelos/`
**Objetivo:** evaluar calidad (silueta), **estabilidad** (ARI), interpretar los grupos y comprobar que
no estén separando por *tipo de prenda* en vez de por *tipo de queja*.

> **Independencia del cuaderno:** no reutilizamos variables en memoria del cuaderno 03. Recalculamos la
> matriz de características con `vectorizador.transform()` (y `svd.transform()` si aplica) sobre el CSV
> limpio. `random_state=42` donde corresponde.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

SEMILLA = 42


def encontrar_raiz() -> Path:
    actual = Path.cwd()
    for candidata in [actual, *actual.parents]:
        if (candidata / "datos").is_dir() and (candidata / "modelos").is_dir():
            return candidata
    return actual


RAIZ = encontrar_raiz()
RUTA_LIMPIAS = RAIZ / "datos" / "procesados" / "resenas_limpias.csv"
DIR_MODELOS = RAIZ / "modelos"

# Cargar modelos persistidos
vectorizador = joblib.load(DIR_MODELOS / "vectorizador.joblib")
kmeans = joblib.load(DIR_MODELOS / "modelo_kmeans.joblib")
ruta_svd = DIR_MODELOS / "svd_lsa.joblib"
reductor = joblib.load(ruta_svd) if ruta_svd.exists() else None
USAR_SVD = reductor is not None
print("Modelos cargados. ¿Pipeline con SVD?:", USAR_SVD)
print("K del modelo:", kmeans.n_clusters)

Modelos cargados. ¿Pipeline con SVD?: True
K del modelo: 6


## 1. Recalcular las características (independencia del cuaderno)

Transformamos el texto limpio con el **vectorizador ya ajustado** (`.transform`, no `.fit`) y, si el
pipeline usa SVD, aplicamos `svd.transform`. Así reproducimos exactamente el espacio en que se entrenó
K-means, sin depender de variables de otro cuaderno.

In [2]:
df = pd.read_csv(RUTA_LIMPIAS)
df = df.dropna(subset=["texto_limpio"]).reset_index(drop=True)
df = df[df["texto_limpio"].str.strip().str.len() > 0].reset_index(drop=True)

X_tfidf = vectorizador.transform(df["texto_limpio"])
X_final = reductor.transform(X_tfidf) if USAR_SVD else X_tfidf
print("Matriz TF-IDF:", X_tfidf.shape, "| Matriz final (clustering):", X_final.shape)

# Asignar cada reseña a su clúster con el modelo cargado
df["cluster"] = kmeans.predict(X_final)
print("\nTamaño de cada clúster:")
print(df["cluster"].value_counts().sort_index())

Matriz TF-IDF: (4675, 5000) | Matriz final (clustering): (4675, 100)



Tamaño de cada clúster:
cluster
0    1873
1     435
2     361
3     300
4     720
5     986
Name: count, dtype: int64


## 2. Coeficiente de silueta del modelo final

In [3]:
sil = silhouette_score(X_final, df["cluster"], sample_size=min(4000, X_final.shape[0]), random_state=SEMILLA)
print(f"Coeficiente de silueta (modelo final): {sil:.4f}")
print("Recordatorio: en texto disperso de alta dimensión una silueta ~0.01-0.05 es normal;")
print("sirve para comparar configuraciones, no como nota absoluta de calidad.")

Coeficiente de silueta (modelo final): 0.0216
Recordatorio: en texto disperso de alta dimensión una silueta ~0.01-0.05 es normal;
sirve para comparar configuraciones, no como nota absoluta de calidad.


## 3. Estabilidad con Adjusted Rand Index (ARI)

Medimos cuán **estable** es la partición ante cambios de inicialización. Entrenamos **10 K-means** con
semillas distintas y **`n_init=1`** (no 10): con `n_init=10` todas las corridas convergerían a soluciones
parecidas y la estabilidad saldría **artificialmente alta**. Con `n_init=1` cada semilla explora una
sola inicialización, lo que da una medida honesta. Calculamos el **ARI por pares** entre las 10 corridas.

In [4]:
N_CORRIDAS = 10
etiquetas_corridas = []
for s in range(N_CORRIDAS):
    km = KMeans(n_clusters=kmeans.n_clusters, random_state=s, n_init=1)
    etiquetas_corridas.append(km.fit_predict(X_final))

aris = []
for i in range(N_CORRIDAS):
    for j in range(i + 1, N_CORRIDAS):
        aris.append(adjusted_rand_score(etiquetas_corridas[i], etiquetas_corridas[j]))

aris = np.array(aris)
print(f"Pares comparados: {len(aris)}")
print(f"ARI medio: {aris.mean():.4f}")
print(f"Desviación estándar del ARI: {aris.std():.4f}")
print(f"ARI mínimo / máximo: {aris.min():.4f} / {aris.max():.4f}")

Pares comparados: 45
ARI medio: 0.4014
Desviación estándar del ARI: 0.1107
ARI mínimo / máximo: 0.2326 / 0.7190


**Interpretación del ARI.** ARI = 1 indica particiones idénticas; ARI ≈ 0, coincidencia esperable por
azar. Un ARI medio moderado con desviación no despreciable es **coherente** con una silueta baja: los
límites entre temas de queja son difusos, así que distintas inicializaciones mueven algunas reseñas de
frontera entre grupos. Se reporta como **limitación** del enfoque no supervisado sobre este texto.

## 4. Términos representativos y reseñas cercanas a cada centroide

Para cada clúster: (a) los términos con mayor **peso medio TF-IDF** dentro del grupo, y (b) las reseñas
más cercanas al centroide (menor distancia en el espacio de agrupamiento).

In [5]:
vocab = np.array(vectorizador.get_feature_names_out())


def terminos_top_cluster(c, n=12):
    idx = np.where(df["cluster"].values == c)[0]
    medio = np.asarray(X_tfidf[idx].mean(axis=0)).ravel()
    top = medio.argsort()[::-1][:n]
    return list(vocab[top])


# Distancia de cada reseña a los centroides (en el espacio de agrupamiento)
distancias = kmeans.transform(X_final)

terminos_por_cluster = {}
for c in range(kmeans.n_clusters):
    terminos_por_cluster[c] = terminos_top_cluster(c)
    print(f"\n===== CLÚSTER {c}  (n={int((df['cluster'] == c).sum())}) =====")
    print("Términos top:", ", ".join(terminos_por_cluster[c]))
    idx = np.where(df["cluster"].values == c)[0]
    cercanas = idx[np.argsort(distancias[idx, c])[:4]]  # 4 reseñas más cercanas al centroide
    print("Reseñas cercanas al centroide:")
    for k in cercanas:
        print("   -", str(df.iloc[k]["Review Text"])[:160])


===== CLÚSTER 0  (n=1873) =====
Términos top: not, fit, color, would, fabric, wear, back, cute, short, one, love, could
Reseñas cercanas al centroide:
   - This top is exactly what i was hoping for except it is missing the drawstring. so disappointing, mailing it back.
   - C'mon. the accent is wrong! a simple google search would prevent this.
   - Horrible fit. i do not understand why they but a aline dress with a skin non aline camisole under the dress.
   - I wish there were be fewer asymmetrical tops in the store. i see many tops i like until i discover they have an asymmetrical hem and are longer in the back, whi

===== CLÚSTER 1  (n=435) =====
Términos top: quality, price, poor, not, poor quality, fabric, worth, thin, cheap, not worth, great, material
Reseñas cercanas al centroide:
   - For those who are wondering this sweater coat is 50% lambswool and 50% acrylic. the original version of this coat was 100% lambswool. 50% acrylic yet the price 
   - What a disappointment and for

## 5. Tabla de contingencia: ¿separan por tipo de queja o por tipo de prenda?

Cruzamos los clústeres con `Department Name` y `Class Name` del dataset original. Si cada clúster se
alineara casi perfectamente con un departamento/clase, estaría separando por **tipo de prenda** y no por
**tipo de queja** (sería una limitación, relacionada con las *stopwords de dominio* del cuaderno 03).

In [6]:
cont_dep = pd.crosstab(df["cluster"], df["Department Name"])
print("=== Clúster x Department Name (conteos) ===")
print(cont_dep)
print("\n=== Proporción por fila (cada clúster suma 1) ===")
print((cont_dep.div(cont_dep.sum(axis=1), axis=0)).round(2))

=== Clúster x Department Name (conteos) ===
Department Name  Bottoms  Dresses  Intimate  Jackets  Tops  Trend
cluster                                                          
0                    277      519       143       65   855     14
1                     64      105        30       29   201      6
2                     41      127        20       11   158      4
3                     36       99         8        5   151      1
4                    145      222        54       30   265      4
5                     89      304        35       49   498     11

=== Proporción por fila (cada clúster suma 1) ===
Department Name  Bottoms  Dresses  Intimate  Jackets  Tops  Trend
cluster                                                          
0                   0.15     0.28      0.08     0.03  0.46   0.01
1                   0.15     0.24      0.07     0.07  0.46   0.01
2                   0.11     0.35      0.06     0.03  0.44   0.01
3                   0.12     0.33      0.03    

In [7]:
# Clases más frecuentes para que la tabla sea legible
clases_top = df["Class Name"].value_counts().head(8).index
cont_clase = pd.crosstab(df["cluster"], df["Class Name"])[clases_top]
print("=== Clúster x Class Name (8 clases más frecuentes, conteos) ===")
print(cont_clase)

=== Clúster x Class Name (8 clases más frecuentes, conteos) ===
Class Name  Dresses  Knits  Blouses  Sweaters  Pants  Fine gauge  Jeans  \
cluster                                                                   
0               519    388      252       136    124          79     68   
1               105     94       60        26     23          21     20   
2               127     61       57        27     17          13     12   
3                99     68       49        16     14          18      9   
4               222    142       76        25     57          22     33   
5               304    213      148        88     28          49     24   

Class Name  Skirts  
cluster             
0               56  
1               13  
2                9  
3               11  
4               43  
5               33  


**Lectura.** Si los departamentos/clases aparecen **repartidos** dentro de cada clúster (y no un
clúster = un departamento), es señal de que los grupos capturan **temas transversales de queja**
(talla, tela, color…) más que categorías de producto. Cualquier alineación fuerte con un tipo de prenda
se documenta como **limitación** y se vincula al ajuste de *stopwords de dominio* del cuaderno 03.

## 6. Nombres descriptivos por clúster (interpretación humana posterior)

Asignamos un nombre a cada clúster **leyendo sus términos top y sus reseñas cercanas** (sección 4). Estos
nombres son una **interpretación humana posterior sobre resultados reales**; **no** se usaron para
entrenar el modelo (K-means es no supervisado y solo vio la matriz TF-IDF/SVD).

In [8]:
# Nombres asignados tras inspeccionar términos top y reseñas de cada clúster (sección 4).
# Interpretación humana posterior; NO intervienen en el entrenamiento.
NOMBRES_CLUSTER = {
    0: "Insatisfacción general de ajuste, color y tela",
    1: "Calidad y precio (tela pobre, no vale lo que cuesta)",
    2: "Expectativa vs. realidad ('quería que me gustara')",
    3: "Silueta poco favorecedora (not flattering)",
    4: "Tallaje (la talla corre pequeña o grande)",
    5: "Apariencia vs. modelo (forma holgada / boxy)",
}


def nombre_cluster(c):
    nombre = NOMBRES_CLUSTER.get(c)
    if not nombre or nombre == "PENDIENTE":
        # Respaldo legible si falta un nombre manual
        return "Grupo " + str(c) + ": " + ", ".join(terminos_por_cluster[c][:3])
    return nombre


filas = []
for c in range(kmeans.n_clusters):
    filas.append({
        "cluster": c,
        "nombre": nombre_cluster(c),
        "n_resenas": int((df["cluster"] == c).sum()),
        "terminos_top": ", ".join(terminos_por_cluster[c][:8]),
    })

tabla_final = pd.DataFrame(filas).sort_values("n_resenas", ascending=False).reset_index(drop=True)
pd.set_option("display.max_colwidth", 120)
tabla_final

,cluster,nombre,n_resenas,terminos_top
0,0,"Insatisfacción general de ajuste, color y tela",1873,"not, fit, color, would, fabric, wear, back, cute"
1,5,Apariencia vs. modelo (forma holgada / boxy),986,"look, like, not, looked, look like, fabric, would, boxy"
2,4,Tallaje (la talla corre pequeña o grande),720,"size, small, run, large, not, ordered, fit, run small"
3,1,"Calidad y precio (tela pobre, no vale lo que cuesta)",435,"quality, price, poor, not, poor quality, fabric, worth, thin"
4,2,Expectativa vs. realidad ('quería que me gustara'),361,"wanted, wanted love, love, really wanted, really, not, like, fit"
5,3,Silueta poco favorecedora (not flattering),300,"flattering, not flattering, not, look, fit, fabric, like, would"


## Conclusiones del cuaderno 04

- Se recalcularon las características con `transform()` sobre el CSV limpio, garantizando que el cuaderno
  es **independiente** del 03.
- La **silueta** es baja (normal en texto disperso) y la **estabilidad (ARI)** es moderada: los límites
  entre temas de queja son difusos; se documenta como limitación del enfoque no supervisado.
- La tabla de contingencia permite verificar si los grupos separan por **tipo de queja** (deseado) o por
  **tipo de prenda** (limitación, ligada a las *stopwords de dominio* del cuaderno 03).
- Los nombres de clúster son **interpretación humana posterior** sobre resultados reales y **no** se
  usaron para entrenar el modelo.